# Clase 074 — Early stopping

Aplicamos **early stopping** como regularización implícita en entrenamientos iterativos: monitoreamos la pérdida de validación y detenemos el ajuste cuando deja de mejorar, conservando el mejor modelo. Lo hacemos con `SGDRegressor` (`warm_start` y `early_stopping=True`) y con un loop manual.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset y split 60/20/20

`make_regression` con 50 features y ruido. Separamos train / validación / test y escalamos con `StandardScaler` (SGD es muy sensible a la escala).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

X, y = make_regression(n_samples=2000, n_features=50, noise=20, random_state=42)
X_tmp, Xte, y_tmp, yte = train_test_split(X, y, test_size=0.2, random_state=42)
Xtr, Xva, ytr, yva = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42)  # 60/20/20
sc = StandardScaler().fit(Xtr)
Xtr, Xva, Xte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)

def rmse(y_true, pred):
    return np.sqrt(mean_squared_error(y_true, pred))

print('train', Xtr.shape[0], '| val', Xva.shape[0], '| test', Xte.shape[0])

## 2. Curva manual con `warm_start`

Con `max_iter=1` + `warm_start=True`, cada `fit` continúa desde los pesos previos: reconstruimos la curva de RMSE por época y marcamos la **best epoch** (mínimo de validación).

In [ ]:
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(max_iter=1, warm_start=True, learning_rate='constant',
                   eta0=1e-3, penalty=None, tol=None, random_state=42)
n_epochs = 300
tr_curve, va_curve = [], []
for _ in range(n_epochs):
    sgd.fit(Xtr, ytr)                     # warm_start -> sigue desde los pesos previos
    tr_curve.append(rmse(ytr, sgd.predict(Xtr)))
    va_curve.append(rmse(yva, sgd.predict(Xva)))

best_epoch = int(np.argmin(va_curve))
print('best epoch (min val RMSE):', best_epoch, '| val RMSE:', round(va_curve[best_epoch], 3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tr_curve, label='train')
ax.plot(va_curve, label='validación')
ax.axvline(best_epoch, color='k', ls=':', label='best epoch')
ax.set_xlabel('epoch'); ax.set_ylabel('RMSE'); ax.legend()
ax.set_title('Early stopping: se conserva el mínimo de la val loss')
plt.tight_layout(); plt.show()

## 3. Early stopping automático de sklearn

`SGDRegressor(early_stopping=True)` reserva un `validation_fraction` interno y corta tras `n_iter_no_change` épocas sin mejora. Debe usar menos épocas que el modelo sin early stopping.

In [ ]:
sgd_es = SGDRegressor(early_stopping=True, validation_fraction=0.2,
                      n_iter_no_change=10, tol=1e-4, max_iter=1000, random_state=42).fit(Xtr, ytr)
sgd_full = SGDRegressor(early_stopping=False, max_iter=1000, tol=1e-4, random_state=42).fit(Xtr, ytr)
print('early stopping -> n_iter_:', sgd_es.n_iter_, '| RMSE test:', round(rmse(yte, sgd_es.predict(Xte)), 3))
print('sin early stop -> n_iter_:', sgd_full.n_iter_, '| RMSE test:', round(rmse(yte, sgd_full.predict(Xte)), 3))
assert sgd_es.n_iter_ <= sgd_full.n_iter_
print('OK: early stopping corta antes (menos épocas)')

## 4. Efecto de la paciencia

`n_iter_no_change` es la paciencia. Muy baja corta prematuro por fluctuaciones; más alta tolera mesetas ruidosas.

In [ ]:
print('n_iter_no_change | épocas (n_iter_) | RMSE test')
for patience in [1, 5, 20, 100]:
    mdl = SGDRegressor(early_stopping=True, validation_fraction=0.2,
                       n_iter_no_change=patience, tol=1e-4, max_iter=2000,
                       random_state=42).fit(Xtr, ytr)
    print(f'{patience:16d} | {mdl.n_iter_:16d} | {rmse(yte, mdl.predict(Xte)):.3f}')

## 5. Implementación manual con snapshot

Un loop con `partial_fit`, `best_loss`, `copy.deepcopy` del mejor modelo y un contador de paciencia. Sin el snapshot, al cortar nos quedaríamos con pesos ya degradados.

In [ ]:
import copy

sgd = SGDRegressor(max_iter=1, warm_start=True, learning_rate='constant',
                   eta0=1e-3, penalty=None, tol=None, random_state=42)
best_loss, best_model, patience, wait = np.inf, None, 15, 0
for epoch in range(500):
    sgd.fit(Xtr, ytr)
    val_loss = rmse(yva, sgd.predict(Xva))
    if val_loss < best_loss - 1e-4:
        best_loss, best_model, wait = val_loss, copy.deepcopy(sgd), 0
    else:
        wait += 1
        if wait >= patience:
            print(f'stop en epoch {epoch} (sin mejora en {patience} épocas)')
            break

assert best_model is not None
print('mejor val RMSE:', round(best_loss, 3),
      '| RMSE test del mejor modelo:', round(rmse(yte, best_model.predict(Xte)), 3))

## 6. Comparación con Ridge

Sobre el mismo problema, comparamos early stopping contra Ridge con `alpha` tuneado por CV: dos formas de regularizar.

In [ ]:
from sklearn.linear_model import RidgeCV

ridge = RidgeCV(alphas=np.logspace(-2, 3, 30)).fit(Xtr, ytr)
print(f'Ridge (alpha CV = {ridge.alpha_:.3f}) RMSE test:', round(rmse(yte, ridge.predict(Xte)), 3))
print('SGD + early stopping        RMSE test:', round(rmse(yte, best_model.predict(Xte)), 3))

## Ejercicios

1. **Curva sin escalar.** Repetí la curva manual sin `StandardScaler` y observá cómo desaparece la "U" clara: SGD oscila y la val loss deja de ser legible.
2. **Val vs test.** Cambiá el criterio de corte para que use el test set en lugar de validación y discutí por qué eso filtra información y arruina la estimación de generalización.
3. **Paciencia vs época final.** Graficá `n_iter_` en función de `n_iter_no_change` del ejercicio 4 y comentá la tendencia.
4. **Boosting.** Aplicá la misma idea a `GradientBoostingRegressor` usando `n_iter_no_change` y `validation_fraction`.

## Conclusiones

- Early stopping es **regularización implícita**: limita cuánto se ajustan los pesos, controlando la capacidad efectiva.
- La señal de corte es la **validation loss**, nunca la de train (que siempre baja) ni la de test (que se reserva para el final).
- Sin guardar el **snapshot** del mejor modelo, al cortar nos quedaríamos con pesos ya sobreajustados.
- Brilla en modelos **iterativos** costosos (SGD, boosting, redes); para modelos cerrados como Ridge no hay épocas que detener.